# World Cup Project — Get Data Starter Notebook

This notebook downloads historical international soccer data and saves it into a local `data/` folder.

It creates both raw and cleaned CSV files that can be used in the modeling notebook.


## What data are we getting?

This notebook pulls three public CSV files from the `international_results` GitHub dataset:

1. **Match results**
   - Historical international match results
   - Includes teams, scores, dates, tournament names, match location, and whether the match was played on neutral ground
   - This is the main dataset for modeling match outcomes

2. **Shootouts**
   - Penalty shootout results
   - Useful later if you build a knockout-round World Cup simulator

3. **Goalscorers**
   - Player-level goalscorer data
   - Includes scorer names, minute of goal, own goals, and penalties
   - Not required for the first match model, but useful for later player/team attacking analysis


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

pd.set_option("display.max_columns", 100)

print("Current working directory:")
print(os.getcwd())


c:\Users\conno\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\conno\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Current working directory:
c:\Users\conno\OneDrive\Documents\World Cup Analytics


## 1. Create the data folder

This creates a `data/` folder in the same working directory where your notebook is running.


In [2]:
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder:")
print(DATA_DIR.resolve())


Data folder:
C:\Users\conno\OneDrive\Documents\World Cup Analytics\data


## 2. Download the raw datasets

In [3]:
results_url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
shootouts_url = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"
goalscorers_url = "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv"

results = pd.read_csv(results_url)
shootouts = pd.read_csv(shootouts_url)
goalscorers = pd.read_csv(goalscorers_url)

print("Downloaded rows:")
print("results:", len(results))
print("shootouts:", len(shootouts))
print("goalscorers:", len(goalscorers))


Downloaded rows:
results: 49445
shootouts: 678
goalscorers: 47601


## 3. Preview match results

This is the most important dataset for the first modeling notebook.


In [4]:
results.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [5]:
results.info()


<class 'pandas.DataFrame'>
RangeIndex: 49445 entries, 0 to 49444
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49445 non-null  str    
 1   home_team   49445 non-null  str    
 2   away_team   49445 non-null  str    
 3   home_score  49373 non-null  float64
 4   away_score  49373 non-null  float64
 5   tournament  49445 non-null  str    
 6   city        49445 non-null  str    
 7   country     49445 non-null  str    
 8   neutral     49445 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 5.9 MB


## 4. Preview shootouts

In [6]:
shootouts.head()


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


## 5. Preview goalscorers

In [7]:
goalscorers.head()


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False


## 6. Save raw CSV files

These are saved exactly as downloaded.


In [8]:
results.to_csv(DATA_DIR / "results_raw.csv", index=False)
shootouts.to_csv(DATA_DIR / "shootouts_raw.csv", index=False)
goalscorers.to_csv(DATA_DIR / "goalscorers_raw.csv", index=False)

print("Raw files saved:")
print(DATA_DIR / "results_raw.csv")
print(DATA_DIR / "shootouts_raw.csv")
print(DATA_DIR / "goalscorers_raw.csv")


Raw files saved:
c:\Users\conno\OneDrive\Documents\World Cup Analytics\data\results_raw.csv
c:\Users\conno\OneDrive\Documents\World Cup Analytics\data\shootouts_raw.csv
c:\Users\conno\OneDrive\Documents\World Cup Analytics\data\goalscorers_raw.csv


## 7. Clean the match results

For the starter model, we need clean match-level data.

Cleaning steps:
- Convert `date` to datetime
- Remove rows missing scores or team names
- Standardize team and tournament text columns
- Ensure scores are numeric integers
- Ensure `neutral` is a true/false column
- Create result labels:
  - `home_win`
  - `draw`
  - `away_win`
- Create basic scoring columns:
  - `goal_diff`
  - `total_goals`


In [9]:
results_clean = results.copy()

results_clean["date"] = pd.to_datetime(results_clean["date"], errors="coerce")

text_cols = ["home_team", "away_team", "tournament", "city", "country"]
for col in text_cols:
    if col in results_clean.columns:
        results_clean[col] = results_clean[col].astype(str).str.strip()

results_clean["home_score"] = pd.to_numeric(results_clean["home_score"], errors="coerce")
results_clean["away_score"] = pd.to_numeric(results_clean["away_score"], errors="coerce")

results_clean = results_clean.dropna(
    subset=["date", "home_team", "away_team", "home_score", "away_score"]
).copy()

results_clean["home_score"] = results_clean["home_score"].astype(int)
results_clean["away_score"] = results_clean["away_score"].astype(int)

if "neutral" in results_clean.columns:
    results_clean["neutral"] = results_clean["neutral"].astype(bool)
else:
    results_clean["neutral"] = False

results_clean["result"] = np.select(
    [
        results_clean["home_score"] > results_clean["away_score"],
        results_clean["home_score"] == results_clean["away_score"],
        results_clean["home_score"] < results_clean["away_score"],
    ],
    ["home_win", "draw", "away_win"]
)

results_clean["goal_diff"] = results_clean["home_score"] - results_clean["away_score"]
results_clean["total_goals"] = results_clean["home_score"] + results_clean["away_score"]

results_clean = results_clean.sort_values("date").reset_index(drop=True)

results_clean.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,goal_diff,total_goals
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,draw,0,0
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,home_win,2,6
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,home_win,1,3
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,draw,0,4
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,home_win,3,3


## 8. Clean shootouts

Basic cleaning:
- Convert `date` to datetime
- Strip team names
- Sort by date


In [10]:
shootouts_clean = shootouts.copy()

shootouts_clean["date"] = pd.to_datetime(shootouts_clean["date"], errors="coerce")

for col in ["home_team", "away_team", "winner"]:
    if col in shootouts_clean.columns:
        shootouts_clean[col] = shootouts_clean[col].astype(str).str.strip()

shootouts_clean = shootouts_clean.dropna(subset=["date"]).copy()
shootouts_clean = shootouts_clean.sort_values("date").reset_index(drop=True)

shootouts_clean.head()


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


## 9. Clean goalscorers

Basic cleaning:
- Convert `date` to datetime
- Strip text columns
- Convert minute to numeric
- Ensure own-goal and penalty fields are true/false


In [11]:
goalscorers_clean = goalscorers.copy()

goalscorers_clean["date"] = pd.to_datetime(goalscorers_clean["date"], errors="coerce")

for col in ["home_team", "away_team", "team", "scorer"]:
    if col in goalscorers_clean.columns:
        goalscorers_clean[col] = goalscorers_clean[col].astype(str).str.strip()

if "minute" in goalscorers_clean.columns:
    goalscorers_clean["minute"] = pd.to_numeric(goalscorers_clean["minute"], errors="coerce")

for col in ["own_goal", "penalty"]:
    if col in goalscorers_clean.columns:
        goalscorers_clean[col] = goalscorers_clean[col].astype(bool)

goalscorers_clean = goalscorers_clean.dropna(subset=["date"]).copy()
goalscorers_clean = goalscorers_clean.sort_values("date").reset_index(drop=True)

goalscorers_clean.head()


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False


## 10. Save cleaned CSV files

This is the key section.

The modeling notebook will look for `data/results.csv`, so this notebook saves the cleaned match results under that exact name.

It also saves `results_clean.csv` for clarity.


In [12]:
results_clean.to_csv(DATA_DIR / "results.csv", index=False)
results_clean.to_csv(DATA_DIR / "results_clean.csv", index=False)

shootouts_clean.to_csv(DATA_DIR / "shootouts.csv", index=False)
shootouts_clean.to_csv(DATA_DIR / "shootouts_clean.csv", index=False)

goalscorers_clean.to_csv(DATA_DIR / "goalscorers.csv", index=False)
goalscorers_clean.to_csv(DATA_DIR / "goalscorers_clean.csv", index=False)

print("Cleaned files saved to:")
print(DATA_DIR.resolve())

print("\nSaved CSV files:")
for file in sorted(DATA_DIR.glob("*.csv")):
    print("-", file.name)


Cleaned files saved to:
C:\Users\conno\OneDrive\Documents\World Cup Analytics\data

Saved CSV files:
- goalscorers.csv
- goalscorers_clean.csv
- goalscorers_raw.csv
- results.csv
- results_clean.csv
- results_raw.csv
- shootouts.csv
- shootouts_clean.csv
- shootouts_raw.csv


## 11. Verify the files exist

Run this cell to make sure the CSVs were actually saved where you expect.


In [13]:
expected_files = [
    "results.csv",
    "results_clean.csv",
    "shootouts.csv",
    "shootouts_clean.csv",
    "goalscorers.csv",
    "goalscorers_clean.csv",
]

for filename in expected_files:
    path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")


results.csv: FOUND
results_clean.csv: FOUND
shootouts.csv: FOUND
shootouts_clean.csv: FOUND
goalscorers.csv: FOUND
goalscorers_clean.csv: FOUND


## 12. Quick data summary

In [14]:
print("Match results date range:")
print(results_clean["date"].min(), "to", results_clean["date"].max())

print("\nNumber of matches:", len(results_clean))
print("Number of teams:", pd.concat([results_clean["home_team"], results_clean["away_team"]]).nunique())

print("\nResult distribution:")
print(results_clean["result"].value_counts(normalize=True).round(3))


Match results date range:
1872-11-30 00:00:00 to 2026-06-07 00:00:00

Number of matches: 49373
Number of teams: 336

Result distribution:
result
home_win    0.490
away_win    0.283
draw        0.227
Name: proportion, dtype: float64
